# Dashboard: Contour Panels with Overlaid Sets

This board puts `nb.contour` into the interactive dashboard. A contour panel is
like any other panel, with two specifics:

- **`z`** &mdash; the column mapped to color (contours *require* a z). It is a
  first-class panel key with its own dropdown, shown only for contour panels.
- **`overlay_sets`** (via the panel's **`kwargs`** passthrough) &mdash; other datasets
  whose `(x, y)` points are drawn on top of the field, each honoring its own style
  (a set with a linestyle draws as a line, otherwise as markers).

The contour surface is built from the datasets the panel plots. Normally that is
the board-wide selection in the **header picker**; a contour panel usually **pins**
its surface set instead (`datasets=[0]`), so the map stays intact whatever the
header selects. `overlay_sets` are drawn over it regardless of selection.

> A contour panel with no `z` shows a graceful in-panel error rather than crashing
> the board &mdash; handy if you switch a non-contour panel to `contour` from the
> plot-type dropdown.

## 0. Setup &mdash; a compressor efficiency map + overlays

Three datasets:

| Set | Role | Drawn as |
|---|---|---|
| 0 &mdash; Efficiency field | scattered `(FLOW, PR, EFF, MARGIN)` samples | the contour surface |
| 1 &mdash; Surge line | a boundary curve in `(FLOW, PR)` | a line overlay |
| 2 &mdash; Operating points | discrete `(FLOW, PR)` test points | marker overlay |

We style sets 1 and 2 so the overlay reads clearly: the surge line as a solid line,
the operating points as circles.

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(0)

# Set 0 -- the EFFICIENCY FIELD: scattered (FLOW, PR) samples with two z surfaces.
# EFF peaks near (FLOW=5, PR=2.6); MARGIN falls off with PR.
flow = rng.uniform(2.0, 8.0, 220)
pr   = rng.uniform(1.5, 4.0, 220)
field = pd.DataFrame({
    'FLOW': flow,
    'PR':   pr,
    'EFF':    0.88 - 0.012 * (flow - 5.0) ** 2 - 0.06 * (pr - 2.6) ** 2,
    'MARGIN': 4.0 - pr,
})

# Set 1 -- a SURGE LINE boundary, ordered so it draws as a clean curve.
sx = np.linspace(2.2, 7.5, 12)
surge = pd.DataFrame({'FLOW': sx, 'PR': 1.6 + 0.42 * sx})

# Set 2 -- measured OPERATING POINTS: a few discrete (FLOW, PR) points.
ops = pd.DataFrame({'FLOW': [3.5, 4.5, 5.0, 5.5, 6.5],
                    'PR':   [2.0, 2.4, 2.7, 3.0, 3.4]})

uc = UnichartNotebook()
uc.load(field, title='Efficiency field')    # Set 0  (the surface)
uc.load(surge, title='Surge line')          # Set 1  (line overlay)
uc.load(ops,   title='Operating points')    # Set 2  (marker overlay)

# Style the overlays (each overlay set keeps its own style on the contour).
uc.color(1, 'black'); uc.linestyle(1, '-')                  # surge -> solid line
uc.color(2, 'red');   uc.marker(2, 'o'); uc.markersize(2, 13)  # ops -> red circles
uc.sets[0].df.head()

## 1. The contour board

Two contour panels over the same field &mdash; efficiency and surge margin &mdash; each with
the surge line and operating points overlaid. The panel spec:

- `z='EFF'` / `z='MARGIN'` &rarr; the color column. A **`z` dropdown** appears in the
  control row for contour panels (and is hidden for other plot types), so you can
  swap the surface live.
- `datasets=[0]` &rarr; the panel is **pinned**: the contour surface always comes from
  Set 0, ignoring the header's dataset picker (note the "pinned" badge on the card).
- `kwargs={'overlay_sets': [1, 2]}` &rarr; the sets drawn on top.

Everything stays live: change a panel's **x/y/z**, edit its **title** in the card
header, flip the board **theme**, or switch the **plot type** from the dropdowns.
(The z dropdown shows only while the type is `contour`; switching away hides it and
drops the contour-only kwargs automatically.)

In [ ]:
uc.dashboard(
    panels=[
        {'method': 'contour', 'x': 'FLOW', 'y': 'PR', 'z': 'EFF',
         'suptitle': 'Efficiency map + surge + ops',
         'datasets': [0],
         'kwargs': {'overlay_sets': [1, 2]}},
        {'method': 'contour', 'x': 'FLOW', 'y': 'PR', 'z': 'MARGIN',
         'suptitle': 'Surge margin',
         'datasets': [0],
         'kwargs': {'overlay_sets': [1, 2]}},
    ],
    title='Compressor map',
    ncols=2,
    width=560,
    height=440,
)

## 2. Notes

- **`overlay_sets`** accepts a dataset index, a list of indices, or `'all'`. Overlay
  sets are drawn even when they aren't among the datasets the panel plots.
- **Overlay appearance** follows each set's own style (`color`, `marker`,
  `markersize`, `linestyle`, ...): set a `linestyle` to draw a connected line
  (good for a boundary), leave it off to show markers.
- **Other contour kwargs** pass through the same way, e.g.
  `kwargs={'overlay_sets': [1, 2], 'colorscale': 'Viridis',
  'ncontours': 15, 'contours_coloring': 'lines'}`.
- **No z** &rarr; the panel shows an in-panel error message instead of breaking the
  board. Seed `z` in the panel spec for any panel you want to start as a contour.
- **Export data.** Each panel's **⬇ csv** button downloads its CSV (for a contour,
  the surface grid as long `x, y, z` rows plus the overlay points); the header's
  **⬇ export all panels** combines every panel into one CSV.

See `dashboard_demo.ipynb` for the general (non-contour) dashboard walkthrough.